# 📊 07 - Model Karşılaştırma Raporu

## 4 Modelin Detaylı Analizi

Bu notebook, eğitilmiş 4 modeli karşılaştırır:
1. **Baseline CNN** (sıfırdan eğitim) — Kıyaslama noktası
2. **MobileNetV2** (transfer learning) — Hafif
3. **EfficientNetB0** (transfer learning) — Dengeli
4. **ResNet50** (transfer learning) — En doğru

### Karşılaştırma Yöntemleri
1. ✅ Genel metrik tablosu
2. ✅ Test accuracy bar grafiği
3. ✅ F1-score karşılaştırması
4. ✅ Model boyutu analizi
5. ✅ Trade-off grafiği
6. ✅ Confusion matrix
7. ✅ Sınıf bazlı F1 karşılaştırması

### 🔬 Bilimsel Bulgu
**Transfer learning katkısı: +%12.89 puan** iyileşme  
(Baseline: %85.76 → ResNet50: %98.65)

In [ ]:
# ============================================================
# 1. HAZIRLIK - 4 modeli yükle
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficient_preprocess

IMG_SIZE = 224
BATCH_SIZE = 32
drive_proje = "/content/drive/MyDrive/Domates_Projesi"

# Veriyi Drive'dan kopyala
if not os.path.exists("tomato_data"):
    shutil.copytree(f"{drive_proje}/data", "tomato_data")

# Modelleri Drive'dan kopyala
os.makedirs("models", exist_ok=True)
for model_file in os.listdir(f"{drive_proje}/models"):
    src = f"{drive_proje}/models/{model_file}"
    dst = f"models/{model_file}"
    if not os.path.exists(dst):
        shutil.copy(src, dst)

# 4 modeli yükle
print("📥 Modeller yükleniyor...")
baseline_model = load_model('models/baseline_cnn.keras')
mobilenet_model = load_model('models/mobilenetv2_final.keras')
resnet_model = load_model('models/resnet50_final.keras')
eff_model = load_model('models/efficientnetb0_final.keras')
print("✅ 4 model yüklendi")

In [ ]:
# ============================================================
# 2. HER MODEL İÇİN ÖZEL TEST GENERATOR
# ============================================================

# Baseline CNN için (rescale=1./255)
test_datagen_baseline = ImageDataGenerator(rescale=1./255)
test_generator_baseline = test_datagen_baseline.flow_from_directory(
    "tomato_data/test", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# MobileNetV2 için
test_datagen_mobile = ImageDataGenerator(rescale=1./255)
test_generator_mobile = test_datagen_mobile.flow_from_directory(
    "tomato_data/test", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# ResNet50 için
test_datagen_resnet = ImageDataGenerator(preprocessing_function=resnet_preprocess)
test_generator_resnet = test_datagen_resnet.flow_from_directory(
    "tomato_data/test", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# EfficientNetB0 için
test_datagen_eff = ImageDataGenerator(preprocessing_function=efficient_preprocess)
test_generator_eff = test_datagen_eff.flow_from_directory(
    "tomato_data/test", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print("✅ 4 model için test generator hazır")

In [ ]:
# ============================================================
# 3. 4 MODELİN TEST PERFORMANSI
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score

print("=" * 70)
print("🧪 4 MODEL TEST DEĞERLENDİRMESİ")
print("=" * 70)

# Baseline CNN
print("\n⚪ Baseline CNN değerlendiriliyor...")
test_generator_baseline.reset()
loss_baseline, acc_baseline = baseline_model.evaluate(test_generator_baseline, verbose=0)
test_generator_baseline.reset()
pred_baseline = baseline_model.predict(test_generator_baseline, verbose=0)
y_pred_baseline = np.argmax(pred_baseline, axis=1)
y_true = test_generator_baseline.classes
prec_baseline = precision_score(y_true, y_pred_baseline, average='weighted')
rec_baseline = recall_score(y_true, y_pred_baseline, average='weighted')
f1_baseline = f1_score(y_true, y_pred_baseline, average='weighted')
size_baseline = os.path.getsize('models/baseline_cnn.keras') / (1024 * 1024)

# MobileNetV2
print("🔵 MobileNetV2 değerlendiriliyor...")
test_generator_mobile.reset()
loss_mobile, acc_mobile = mobilenet_model.evaluate(test_generator_mobile, verbose=0)
test_generator_mobile.reset()
pred_mobile = mobilenet_model.predict(test_generator_mobile, verbose=0)
y_pred_mobile = np.argmax(pred_mobile, axis=1)
prec_mobile = precision_score(y_true, y_pred_mobile, average='weighted')
rec_mobile = recall_score(y_true, y_pred_mobile, average='weighted')
f1_mobile = f1_score(y_true, y_pred_mobile, average='weighted')
size_mobile = os.path.getsize('models/mobilenetv2_final.keras') / (1024 * 1024)

# ResNet50
print("🔴 ResNet50 değerlendiriliyor...")
test_generator_resnet.reset()
loss_resnet, acc_resnet = resnet_model.evaluate(test_generator_resnet, verbose=0)
test_generator_resnet.reset()
pred_resnet = resnet_model.predict(test_generator_resnet, verbose=0)
y_pred_resnet = np.argmax(pred_resnet, axis=1)
prec_resnet = precision_score(y_true, y_pred_resnet, average='weighted')
rec_resnet = recall_score(y_true, y_pred_resnet, average='weighted')
f1_resnet = f1_score(y_true, y_pred_resnet, average='weighted')
size_resnet = os.path.getsize('models/resnet50_final.keras') / (1024 * 1024)

# EfficientNetB0
print("🟢 EfficientNetB0 değerlendiriliyor...")
test_generator_eff.reset()
loss_eff, acc_eff = eff_model.evaluate(test_generator_eff, verbose=0)
test_generator_eff.reset()
pred_eff = eff_model.predict(test_generator_eff, verbose=0)
y_pred_eff = np.argmax(pred_eff, axis=1)
prec_eff = precision_score(y_true, y_pred_eff, average='weighted')
rec_eff = recall_score(y_true, y_pred_eff, average='weighted')
f1_eff = f1_score(y_true, y_pred_eff, average='weighted')
size_eff = os.path.getsize('models/efficientnetb0_final.keras') / (1024 * 1024)

print("\n✅ 4 model değerlendirmesi tamamlandı")

In [ ]:
# ============================================================
# 4. KARŞILAŞTIRMA TABLOSU
# ============================================================

import pandas as pd

results_data = {
    'Model': ['Baseline CNN', 'MobileNetV2', 'ResNet50', 'EfficientNetB0'],
    'Yaklaşım': ['Sıfırdan eğitim', 'Transfer learning', 'Transfer learning', 'Transfer learning'],
    'Test Accuracy (%)': [acc_baseline*100, acc_mobile*100, acc_resnet*100, acc_eff*100],
    'F1-Score': [f1_baseline, f1_mobile, f1_resnet, f1_eff],
    'Precision': [prec_baseline, prec_mobile, prec_resnet, prec_eff],
    'Recall': [rec_baseline, rec_mobile, rec_resnet, rec_eff],
    'Boyut (MB)': [size_baseline, size_mobile, size_resnet, size_eff]
}

df_results = pd.DataFrame(results_data)
print("=" * 100)
print("📊 4 MODEL KARŞILAŞTIRMA TABLOSU")
print("=" * 100)
print(df_results.to_string(index=False))
print("=" * 100)

# Transfer learning katkısı
improvement = acc_resnet*100 - acc_baseline*100
print(f"\n🔬 Transfer learning katkısı: +%{improvement:.2f} puan iyileşme")
print(f"   (Baseline %{acc_baseline*100:.2f} → ResNet50 %{acc_resnet*100:.2f})")

df_results.to_csv('karsilastirma_tablosu.csv', index=False)

In [ ]:
# ============================================================
# 5. BAR GRAFİKLERİ — 4 MODEL
# ============================================================

import matplotlib.pyplot as plt

models = ['Baseline\nCNN', 'MobileNetV2', 'ResNet50', 'EfficientNetB0']
test_accs = [acc_baseline*100, acc_mobile*100, acc_resnet*100, acc_eff*100]
f1_scores = [f1_baseline, f1_mobile, f1_resnet, f1_eff]
sizes = [size_baseline, size_mobile, size_resnet, size_eff]
colors = ['#95a5a6', '#3498db', '#e74c3c', '#2ecc71']  # gri = baseline

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Test Accuracy
bars1 = axes[0].bar(models, test_accs, color=colors, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Test Accuracy Karşılaştırması', fontsize=13, fontweight='bold')
axes[0].set_ylim(80, 100)
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.2f}%', ha='center', fontsize=11, fontweight='bold')

# F1-Score
bars2 = axes[1].bar(models, f1_scores, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('F1-Score', fontsize=12, fontweight='bold')
axes[1].set_title('F1-Score Karşılaştırması', fontsize=13, fontweight='bold')
axes[1].set_ylim(0.8, 1.0)
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

# Boyut
bars3 = axes[2].bar(models, sizes, color=colors, edgecolor='black', linewidth=1.5)
axes[2].set_ylabel('Model Boyutu (MB)', fontsize=12, fontweight='bold')
axes[2].set_title('Model Boyutu', fontsize=13, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)
for bar, val in zip(bars3, sizes):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{val:.1f} MB', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('🏆 4 Model Performans Karşılaştırması', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('metrik_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 6. TRADE-OFF GRAFİĞİ
# ============================================================

fig, ax = plt.subplots(figsize=(12, 8))
labels = ['Baseline CNN', 'MobileNetV2', 'ResNet50', 'EfficientNetB0']

for i, (label, size, acc) in enumerate(zip(labels, sizes, test_accs)):
    ax.scatter(size, acc, s=400, c=colors[i], edgecolors='black', linewidths=2,
               label=label, alpha=0.8, zorder=3)
    ax.annotate(f'{label}\n({acc:.2f}%, {size:.1f}MB)',
                xy=(size, acc), xytext=(15, 10),
                textcoords='offset points', fontsize=11, fontweight='bold')

ax.axhspan(95, 100, alpha=0.1, color='green', label='İdeal Bölge')
ax.set_xlabel('Model Boyutu (MB)', fontsize=13, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('🎯 Trade-off Analizi: Accuracy vs Model Boyutu', fontsize=14, fontweight='bold')
ax.set_ylim(80, 100)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 7. CONFUSION MATRIX — 4 MODEL
# ============================================================

from sklearn.metrics import confusion_matrix
import seaborn as sns

class_names_short = [
    "Bakteriyel", "Erken Yan.", "Geç Yan.", "Yaprak Küfü",
    "Septorya", "Kırmızı Örm.", "Hedef Leke", "Sarı Yap.",
    "Mozaik", "Sağlıklı"
]

fig, axes = plt.subplots(2, 2, figsize=(20, 16))

cm_data = [
    (axes[0, 0], y_true, y_pred_baseline, 'Baseline CNN', 'Greys'),
    (axes[0, 1], y_true, y_pred_mobile, 'MobileNetV2', 'Blues'),
    (axes[1, 0], y_true, y_pred_resnet, 'ResNet50', 'Reds'),
    (axes[1, 1], y_true, y_pred_eff, 'EfficientNetB0', 'Greens')
]

for ax, yt, yp, model_name, cmap in cm_data:
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, cbar=False,
                xticklabels=class_names_short, yticklabels=class_names_short, ax=ax,
                annot_kws={'size': 9})
    ax.set_title(f'{model_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Tahmin', fontsize=11)
    ax.set_ylabel('Gerçek', fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('🔥 4 Model Confusion Matrix Karşılaştırması', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('confusion_matrix_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 8. SINIF BAZLI F1-SCORE KARŞILAŞTIRMA
# ============================================================

f1_baseline_class = f1_score(y_true, y_pred_baseline, average=None)
f1_mobile_class = f1_score(y_true, y_pred_mobile, average=None)
f1_resnet_class = f1_score(y_true, y_pred_resnet, average=None)
f1_eff_class = f1_score(y_true, y_pred_eff, average=None)

x = np.arange(len(class_names_short))
width = 0.20

fig, ax = plt.subplots(figsize=(18, 7))

ax.bar(x - 1.5*width, f1_baseline_class, width, label='Baseline CNN', color='#95a5a6', edgecolor='black')
ax.bar(x - 0.5*width, f1_mobile_class, width, label='MobileNetV2', color='#3498db', edgecolor='black')
ax.bar(x + 0.5*width, f1_resnet_class, width, label='ResNet50', color='#e74c3c', edgecolor='black')
ax.bar(x + 1.5*width, f1_eff_class, width, label='EfficientNetB0', color='#2ecc71', edgecolor='black')

ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_xlabel('Hastalık Sınıfı', fontsize=12, fontweight='bold')
ax.set_title('🎯 Sınıf Bazlı F1-Score Karşılaştırması (4 Model)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names_short, rotation=45, ha='right')
ax.legend(loc='lower right', fontsize=11)
ax.set_ylim(0.6, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## ✅ Karşılaştırma Tamamlandı

### 🏆 Sonuç Özeti

| Model | Test Acc | F1 | Boyut | Yaklaşım |
|-------|----------|-----|-------|----------|
| Baseline CNN | %85.76 | 0.8571 | 39.61 MB | Sıfırdan |
| MobileNetV2 | %92.40 | 0.9242 | 22.74 MB | Transfer learning |
| EfficientNetB0 | %96.55 | 0.9657 | 29.58 MB | Transfer learning |
| **ResNet50** ⭐ | **%98.65** | **0.9865** | 203.90 MB | Transfer learning |

### 🎯 Bilimsel Bulgular
1. **Transfer learning'in katkısı kanıtlandı**: +%12.89 iyileşme
2. **Baseline iyi bir referans**: %85.76 problemin öğrenilebilir olduğunu gösterdi
3. **EfficientNetB0 dengeli**: Mobil için ideal (%96.55, 30 MB)
4. **ResNet50 maksimum**: En yüksek doğruluk ama 7x büyük

### Sıradaki Adım
👉 `08_gradcam_analizi.ipynb` — Grad-CAM ile XAI analizi